# Session 14: Practice Quiz

**Notebook 2 of 2: Self-check** · about 30 minutes

For each question:

1. Write down your answer **before** opening "Show answer", or before running the code cell.
2. Run the cell where there is one.
3. Compare, and make sure you understand the *reasoning* in the answer, not only the result.

This quiz is formative: it's for you, and it isn't graded. It runs offline, with no model and no network.

In [ ]:
import hashlib
import logging
import tempfile
from pathlib import Path

print("ready")

## Q1 · Predict: what does each file become?

The index holds three sources, with the hashes their chunks were built from. The folder now holds three files, with their current hashes:

```python
indexed = {"a.md": "h1", "b.md": "h2", "d.pdf": "h4"}   # store.sources()
on_disk = {"a.md": "h1", "b.md": "h3", "c.md": "h5"}    # source -> content_hash(path)
```

`c.md` turns out to be an empty file. For each of `a.md`, `b.md`, `c.md` and `d.pdf`: which outcome does the run report (added, updated, unchanged, removed, failed), and how many files are sent to the embedder?

<details>
<summary>Show answer</summary>

| Source | Outcome | Why |
|---|---|---|
| `a.md` | **unchanged** | Same hash: not read, not embedded |
| `b.md` | **updated** | Indexed before, different hash: loaded, chunked, embedded, then its chunks replaced |
| `c.md` | **failed** | New, so it's loaded, but an empty file raises `DocumentError`. Nothing is added for it, and the next run tries again |
| `d.pdf` | **removed** | Indexed, but no longer on disk |

**One** file reaches the embedder: `b.md`. `c.md` fails before embedding, and `a.md` is skipped on its hash alone. That's the point of incremental ingestion: the cost of a run follows what changed, not the size of the folder.

</details>

## Q2 · Predict: a new laptop

You clone the documentation repository onto a new laptop, copy the `index/` folder from your old laptop, and run `knowledge-assistant ingest`.

1. What does the run report if the index used **content hashes**?
2. What would it report if the index had used **modification times** instead?
3. The new laptop runs Windows, and Git there is configured to convert line endings to `\r\n` on checkout. Run the cell below, then revise answer 1.

In [ ]:
lf = b"# Runbook\n\nRestart the service.\n"
crlf = lf.replace(b"\n", b"\r\n")
print(hashlib.sha256(lf).hexdigest()[:12], hashlib.sha256(crlf).hexdigest()[:12])

<details>
<summary>Show answer</summary>

1. **Everything unchanged.** A clone recreates the same bytes, so every hash matches, and nothing is embedded.
2. **Everything changed.** A clone writes every file now, so every modification time is newer than the index's. Everything would be re-embedded, for nothing.
3. **Every Markdown file updated, once.** The bytes differ (`\r\n` instead of `\n`), so the hashes differ. PDFs are binary, and Git doesn't convert them. After one run, the index records the new hashes, and later runs are quiet again.

That last case is a "false changed", and it's the cheap kind of mistake: one unnecessary re-embedding. The expensive kind is a "false unchanged", where an edit never reaches the index. Content hashes make the first kind occasionally and the second kind practically never. If the re-embedding matters (a paid API, a huge corpus), configure Git with a `.gitattributes` file so line endings are consistent, rather than weakening the fingerprint.

</details>

## Q3 · Debugging: the outage that erased a runbook

A teammate's `ingest()` handles a changed file like this:

```python
store.remove(source)                                        # clear the old version
vectors = embedder.embed([c.embedding_text for c in chunks])
store.replace(source, chunks, vectors)
```

During one run, the embedding API times out while processing `runbooks/database-failover.md`, and the run stops with an exception. The on-call engineer then asks the assistant how to fail over the orders database.

1. What does retrieval find for that question until the next successful run?
2. What does the next run do with that file, and is the index then correct?
3. Fix the code.

<details>
<summary>Show answer</summary>

1. **Nothing from the failover runbook.** `remove` ran, then `embed` raised, so the runbook had no chunks at all, at exactly the moment someone might need it. Earlier files are fine; the damage is to the file in progress.
2. The next run sees `runbooks/database-failover.md` on disk but **not** in the index, so it's treated as **added**, embedded and stored. The index is correct again, but only after a run succeeds.
3. Do all the work that can fail **before** touching the index:

```python
vectors = embedder.embed([c.embedding_text for c in chunks])  # may raise: nothing changed yet
store.replace(source, chunks, vectors)                        # now swap old for new
```

`replace` is what removes the old chunks, so the separate `remove` disappears. If `embed` raises, the previous version stays searchable. This is the same idea as writing to a temporary file and renaming it over the old one: prepare everything, then switch.

</details>

## Q4 · Debugging: chunks that don't come back

A `ChunkStore.chunks()` implementation reads metadata like this:

```python
Chunk(
    text=text,
    source=metadata["source"],
    title=metadata["title"],
    section=metadata["section"],
    page=metadata["page"],
    index=metadata["index"],
    content_hash=metadata["content_hash"],
)
```

Storing works. Reading back a Markdown file's chunks fails with `KeyError: 'page'`, and reading a PDF's chunks fails with `KeyError: 'section'`. Explain, and fix it.

<details>
<summary>Show answer</summary>

Chroma metadata can't hold `None`. A Markdown chunk has `page=None`, and a PDF chunk has `section=None`, so the writing side either left those keys out deliberately, or used `upsert`, which treats `None` as "remove this key". Either way, the key **isn't there** when you read it back, and `metadata["page"]` raises `KeyError`.

Read the optional fields with `.get()`, which returns `None` for a missing key:

```python
section=metadata.get("section"),
page=metadata.get("page"),
```

Keep `metadata["source"]` and the other required fields as `[...]`: if one of *those* is missing, something is genuinely wrong, and a `KeyError` is the right signal.

The workbook's store tests write chunks with `None` fields and compare what comes back with `==`, which catches this immediately.

</details>

## Q5 · Implementation: a canary test

Here's a function that summarises a document for an operations dashboard. Write a pytest test proving that it never logs the document's contents, at any level. (`caplog` and `tmp_path` are pytest fixtures.) The cell below runs the function so you can see what it does.

In [ ]:
logger = logging.getLogger("dashboard")


def summarise(path: Path) -> dict[str, int | str]:
    text = path.read_text(encoding="utf-8")
    words = len(text.split())
    logger.debug("summarised %s: %d words", path.name, words)
    return {"source": path.name, "words": words}


demo = Path(tempfile.mkdtemp()) / "policy.md"
demo.write_text("# Policy\n\nRotate keys every 90 days.", encoding="utf-8")
print(summarise(demo))

<details>
<summary>Show answer</summary>

```python
import logging

CANARY = "CANARY-3c9e this must never be logged"


def test_summarise_never_logs_the_document(tmp_path, caplog):
    caplog.set_level(logging.DEBUG)  # capture everything, not only warnings
    path = tmp_path / "policy.md"
    path.write_text(f"# Policy\n\n{CANARY}\n", encoding="utf-8")

    result = summarise(path)

    assert result["source"] == "policy.md"  # the function did run
    assert "policy.md" in caplog.text  # and it did log...
    assert "CANARY" not in caplog.text  # ...but not the contents
```

What makes it a good test:

- **`DEBUG` level.** Leaks often hide in debug logging, which is switched on precisely when something is going wrong in production.
- **A unique canary.** Searching the log for an ordinary word like "keys" could match by accident.
- **Positive checks too.** Without `"policy.md" in caplog.text`, a function that logs nothing at all would pass, and so would a broken logging setup.
- **Make it fail once:** add `logger.debug("text: %s", text)` to `summarise` and check that the test catches it. A test you've never seen fail hasn't proven anything.

</details>

## Q6 · Engineering judgment: what should a broken file do to the index?

`policies/access-requests.md` has been indexed for months. Today, an editor saves it with a broken encoding, so the next ingestion run can't read it. A colleague proposes:

> *"When a file fails to load, delete its chunks. We must never serve content that doesn't match the file."*

The capstone keeps the previous chunks instead. Argue both sides, and say when you'd choose the colleague's rule.

<details>
<summary>Show answer</summary>

**Keeping the previous chunks (the capstone's rule):**

- The failure is usually temporary: a half-saved file, an editor problem, a sync in progress. The last good version is almost certainly closer to the truth than no version at all.
- Questions about access requests keep getting answered, with sources, while someone fixes the file.
- The run reports the failure every time, until it's fixed, so it isn't silent.

**Deleting them (the colleague's rule):**

- The index never serves text that no longer matches any readable file.
- It treats "unreadable" like "deleted", which is simple to explain.

**When the colleague is right:** when serving outdated content is worse than serving nothing. One example is a policy that was changed for legal reasons, where the old version must not be quoted. Even then, deletion is better done deliberately: removing the file from the folder (which ingestion already handles), or an explicit `remove` command. An encoding accident shouldn't have that effect.

There's no universally right answer. What matters is that the rule is **chosen, written down, and tested**, rather than being whatever the code happened to do. That's what the workbook's architecture notes are for.

</details>

## Q7 · Comparison: `Embedder` as a Protocol

The capstone declares:

```python
class Embedder(Protocol):
    model_id: str
    dimensions: int

    def embed(self, texts: list[str]) -> np.ndarray: ...
```

`HashingEmbedder` and `FastEmbedEmbedder` never mention `Embedder`. Compare this with an `interface IEmbedder` in Java or C#:

1. How does Python know that `HashingEmbedder` is an `Embedder`?
2. Someone writes an embedder class without a `dimensions` attribute and passes it to `ingest()`. When, and how, does the mistake show up in each language?

<details>
<summary>Show answer</summary>

1. **By its shape** (structural typing). Any object with `model_id`, `dimensions` and an `embed()` method of the right signature fits, whatever it inherits from. In Java or C#, a class is an `IEmbedder` only if it **declares** `implements IEmbedder` (nominal typing). That's why the tests' counting and failing embedders can be small classes that just have the right attributes.
2. **In Java or C#:** at compile time, the class doesn't compile, or can't be passed where an `IEmbedder` is expected. **In Python:** a type checker such as mypy or pyright reports it before you run anything, if you use one (it's an extension in this course). Without one, Python finds out at **run time**, with `AttributeError: ... has no attribute 'dimensions'`, at the first line that reads it. In the capstone, that's probably the configuration check at the start of `ingest()`, before any file is processed.

The trade-off: protocols cost no ceremony and make fakes trivial, but the safety net is a tool you choose to run, plus your tests, rather than the compiler.

</details>